# Get Data from clinicaltrials
La intencion del notebook es levantar en un dataframe local la informacion de la API https://clinicaltrials.gov/data-api/api

## Configuration

In [1]:
import requests
import pandas as pd
import time
import json
from pathlib import Path

BASE_URL   = "https://clinicaltrials.gov/api/v2/studies"
CHECKPOINT = Path("checkpoint.json")   # guarda progreso por si se corta
OUTPUT_DIR = Path("data")

FIELDS = [
    # identificationModule
    "NCTId", "BriefTitle", "OfficialTitle",
    # statusModule
    "OverallStatus", "PrimaryCompletionDate",
    # descriptionModule
    "BriefSummary",
    # conditionsModule
    "Condition", "Keyword",
    # designModule
    "Phase",
    # armsInterventionsModule → armGroups (label + description)
    "ArmGroupLabel", "ArmGroupDescription",
    # eligibilityModule
    "EligibilityCriteria", "MinimumAge", "StdAge",
    # contactsLocationsModule → locations
    "LocationStatus", "LocationFacility", "LocationCity",
    "LocationState", "LocationZip", "LocationCountry", "LocationGeoPoint",
    # derivedSection → conditionBrowseModule → meshes
    "ConditionMeshId", "ConditionMeshTerm",
    # top-level
    "HasResults",
]

STATUSES = [
    "RECRUITING",
    "ACTIVE_NOT_RECRUITING",
    "COMPLETED",
    "NOT_YET_RECRUITING",
    "ENROLLING_BY_INVITATION",
]

## Fetch Data

In [2]:
def fetch_all_studies(page_size: int = 1000) -> list[dict]:
    """Descarga todos los estudios paginando hasta el final."""
    all_studies = []
    page_token  = None

    while True:
        params = {
            "filter.overallStatus": ",".join(STATUSES),
            "fields":               ",".join(FIELDS),
            "pageSize":             page_size,
            "format":               "json",
        }
        if page_token:
            params["pageToken"] = page_token

        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        data = response.json()

        studies = data.get("studies", [])
        if not studies:
            break

        all_studies.extend(studies)
        print(f"Descargados: {len(all_studies)}")

        page_token = data.get("nextPageToken")
        if not page_token:
            break

    return all_studies

## Parse to DF

In [3]:
def parse_study(study: dict) -> dict:
    """Aplana un estudio crudo a un dict plano.
    
    Arrays (conditions, keywords, phases, arm_groups, std_ages,
    locations, condition_meshes) se guardan como JSON string.
    """
    proto       = study.get("protocolSection", {})
    derived     = study.get("derivedSection", {})

    id_mod      = proto.get("identificationModule", {})
    status_mod  = proto.get("statusModule", {})
    desc_mod    = proto.get("descriptionModule", {})
    cond_mod    = proto.get("conditionsModule", {})
    design_mod  = proto.get("designModule", {})
    arms_mod    = proto.get("armsInterventionsModule", {})
    eligib_mod  = proto.get("eligibilityModule", {})
    locs        = proto.get("contactsLocationsModule", {}).get("locations", [])
    cond_browse = derived.get("conditionBrowseModule", {})

    return {
        # identificationModule
        "nct_id":               id_mod.get("nctId"),
        "brief_title":          id_mod.get("briefTitle"),
        "official_title":       id_mod.get("officialTitle"),

        # statusModule
        "overall_status":       status_mod.get("overallStatus"),
        "completion_date":      status_mod.get("primaryCompletionDateStruct", {}).get("date"),

        # descriptionModule
        "brief_summary":        desc_mod.get("briefSummary"),

        # conditionsModule — arrays → JSON
        "conditions":           json.dumps(cond_mod.get("conditions", [])),
        "keywords":             json.dumps(cond_mod.get("keywords", [])),

        # designModule — array → JSON
        "phases":               json.dumps(design_mod.get("phases", [])),

        # armsInterventionsModule → armGroups (label + description) → JSON
        "arm_groups":           json.dumps([
                                    {
                                        "label":       a.get("label"),
                                        "description": a.get("description"),
                                    }
                                    for a in arms_mod.get("armGroups", [])
                                ]),

        # eligibilityModule
        "eligibility_criteria": eligib_mod.get("eligibilityCriteria"),
        "min_age":              eligib_mod.get("minimumAge"),
        "std_ages":             json.dumps(eligib_mod.get("stdAges", [])),

        # contactsLocationsModule → locations → JSON
        "locations":            json.dumps([
                                    {
                                        "status":   l.get("status"),
                                        "zip":      l.get("zip"),
                                        "country":  l.get("country"),
                                        "geoPoint": l.get("geoPoint"),   # {"lat": ..., "lon": ...}
                                        "facility": l.get("facility"),
                                        "city":     l.get("city"),
                                        "state":    l.get("state"),
                                    }
                                    for l in locs
                                ]),

        # derivedSection → conditionBrowseModule → meshes → JSON
        "condition_meshes":     json.dumps([
                                    {
                                        "id":   m.get("id"),
                                        "term": m.get("term"),
                                    }
                                    for m in cond_browse.get("meshes", [])
                                ]),

        # top-level flag
        "has_results":          study.get("hasResults", False),
    }


def to_dataframe(studies: list[dict]) -> pd.DataFrame:
    """Convierte la lista cruda de estudios en un DataFrame."""
    return pd.DataFrame([parse_study(s) for s in studies])


## Use
Fetch → filter oncology → save parquet → upload

In [4]:
import json
from pathlib import Path

# Load oncology terms
_terms_file = Path("oncology_terms.txt")
with open(_terms_file, encoding="utf-8") as _f:
    ONCOLOGY_TERMS = {line.strip().lower() for line in _f if line.strip()}
print(f"Loaded {len(ONCOLOGY_TERMS):,} oncology terms")

def is_oncology(row) -> bool:
    for term in json.loads(row["conditions"] or "[]"):
        if str(term).strip().lower() in ONCOLOGY_TERMS:
            return True
    for term in json.loads(row["keywords"] or "[]"):
        if str(term).strip().lower() in ONCOLOGY_TERMS:
            return True
    for mesh in json.loads(row["condition_meshes"] or "[]"):
        if str(mesh.get("term", "")).strip().lower() in ONCOLOGY_TERMS:
            return True
    return False

# Fetch and parse
raw = fetch_all_studies()
df  = to_dataframe(raw)
print(f"Total estudios: {len(df)}")

# Filter oncology before saving
mask = df.apply(is_oncology, axis=1)
df_onco = df[mask].reset_index(drop=True)
print(f"Oncology studies: {len(df_onco):,} / {len(df):,}")

# Save filtered parquet
df_onco.to_parquet("../data/clinical_trials.parquet", index=False)
print("Saved → ../data/clinical_trials.parquet (oncology only)")

Loaded 26,683 oncology terms
Descargados: 1000
Descargados: 2000
Descargados: 3000
Descargados: 4000
Descargados: 5000
Descargados: 6000
Descargados: 7000
Descargados: 8000
Descargados: 9000
Descargados: 10000
Descargados: 11000
Descargados: 12000
Descargados: 13000
Descargados: 14000
Descargados: 15000
Descargados: 16000
Descargados: 17000
Descargados: 18000
Descargados: 19000
Descargados: 20000
Descargados: 21000
Descargados: 22000
Descargados: 23000
Descargados: 24000
Descargados: 25000
Descargados: 26000
Descargados: 27000
Descargados: 28000
Descargados: 29000
Descargados: 30000
Descargados: 31000
Descargados: 32000
Descargados: 33000
Descargados: 34000
Descargados: 35000
Descargados: 36000
Descargados: 37000
Descargados: 38000
Descargados: 39000
Descargados: 40000
Descargados: 41000
Descargados: 42000
Descargados: 43000
Descargados: 44000
Descargados: 45000
Descargados: 46000
Descargados: 47000
Descargados: 48000
Descargados: 49000
Descargados: 50000
Descargados: 51000
Descargados

## Upload to Hugging Face

In [5]:
from huggingface_hub import HfApi

api = HfApi()

# Crear el dataset repo (solo la primera vez)
#api.create_repo(
#    repo_id="OrneCorsetti/mcd-austral-clinical-trials",
#    repo_type="dataset",
#    private=True  # True = solo vos y quienes invites, False = público
#)

# Subir el archivo
api.upload_file(
    path_or_fileobj=r"D:\GitHub\MCD_Austral_Text_Mining\data\clinical_trials.parquet",
    path_in_repo="clinical_trials.parquet",
    repo_id="OrneCorsetti/mcd-austral-clinical-trials",
    repo_type="dataset"
)

print("✅ Subido a Hugging Face")

c:\Users\orcor\anaconda3\envs\ds_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing Files (1 / 1): 100%|██████████|  216MB /  216MB, 7.06MB/s  
New Data Upload: 100%|██████████|  216MB /  216MB, 7.06MB/s  


✅ Subido a Hugging Face
